**SECTION 2 – Predicting Air Pollution Levels**

Import Libraries

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [ ]:
df = pd.read_csv("urban_traffic_pollution_dataset.csv")

In [ ]:
print(df.head())

         date city_zone  traffic_density  car_count  bus_count  truck_count  \
0  2025-01-01      East              274        291         31           60   
1  2025-01-02      West              434        208         33           38   
2  2025-01-03     North              452        211         36           87   
3  2025-01-04      East              175         88         23           49   
4  2025-01-05      East              179        297         25           50   

   bike_count  temperature   humidity  wind_speed        pm25        no2  \
0          45    34.025600  87.775613   18.861517  151.927012  71.661789   
1         174    30.655508  59.012739    4.813854  250.058331  52.326655   
2         180    18.281122  78.359553    2.430028  148.594255  46.306886   
3          55    15.813154  63.013593    3.949410  120.183514  37.247337   
4         115    38.021196  32.604752   17.738498  158.379485  18.070662   

         co  
0  3.300861  
1  4.015948  
2  3.102343  
3  1.161332 

1. Pollution Variable and Influencing Features

Target variable (Pollution level to predict)

In [ ]:
target = 'pm25'

print("Target Variable:", target)

Target Variable: pm25


Features influencing pollution

In [ ]:
features = [
    'traffic_density',
    'car_count',
    'bus_count',
    'truck_count',
    'bike_count',
    'temperature',
    'humidity',
    'wind_speed'
]

print("\nInput Features:")
print(features)


Input Features:
['traffic_density', 'car_count', 'bus_count', 'truck_count', 'bike_count', 'temperature', 'humidity', 'wind_speed']


Explanation:
*  pm25 represents air pollution level.
*  Traffic and weather conditions directly affect pollution.

**2. Data Preparation for Modeling**

Convert date column

In [ ]:
df['date'] = pd.to_datetime(df['date'], errors='coerce')

Create time-based features

In [ ]:
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

Remove invalid values

In [ ]:
invalid_values = ["error", "missing", "--"]
df.replace(invalid_values, np.nan, inplace=True)

Convert numeric columns

In [ ]:
numeric_cols = [
    'traffic_density',
    'car_count',
    'bus_count',
    'truck_count',
    'bike_count',
    'temperature',
    'humidity',
    'wind_speed',
    'pm25',
    'no2',
    'co'
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce'

Fill missing value

In [ ]:
for col in numeric_cols:
    df[col].fillna(df[col].median(), inplace=True)

Encode categorical column

In [ ]:
encoder = LabelEncoder()
df['city_zone'] = encoder.fit_transform(df['city_zone'])

Final feature set

In [ ]:
X = df[
    [
        'traffic_density',
        'car_count',
        'bus_count',
        'truck_count',
        'bike_count',
        'temperature',
        'humidity',
        'wind_speed',
        'city_zone',
        'month',
        'day'
    ]
]

Target variable

In [ ]:
y = df['pm25']

print("\nDataset Prepared Successfully")

Preprocessing Challenges


1.  Missing values from sensor failure
2. Invalid entries like "error" or "--"
3. Outliers in pollution data
4. Incorrect data types
5. Feature scaling differences
6. Time-series dependency




**3. Different Modeling Techniques**


Split dataset

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

Feature scaling

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

Model 1 : Linear Regression

In [ ]:
lr_model = LinearRegression()

lr_model.fit(X_train_scaled, y_train)

lr_pred = lr_model.predict(X_test_scaled)

Model 2 : Decision Tree

In [ ]:
dt_model = DecisionTreeRegressor(random_state=42)

dt_model.fit(X_train, y_train)

dt_pred = dt_model.predict(X_test)

Model 3 : Random Forest

In [ ]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

Why Some Models Perform Better
* Linear Regression:
 Works well for linear relationships.

* Decision Tree:
 Captures nonlinear patterns but may overfit.

* Random Forest:
 Reduces overfitting and handles complex data better.

**4. Model Evaluation**

In [ ]:
def evaluate_model(name, y_test, predictions):

    mae = mean_absolute_error(y_test, predictions)

    mse = mean_squared_error(y_test, predictions)

    rmse = np.sqrt(mse)

    r2 = r2_score(y_test, predictions)

    print(f"\n{name} Performance")
    print("MAE :", mae)
    print("MSE :", mse)
    print("RMSE:", rmse)
    print("R2 Score:", r2)

Evaluate all models

In [ ]:
evaluate_model("Linear Regression", y_test, lr_pred)

evaluate_model("Decision Tree", y_test, dt_pred)

evaluate_model("Random Forest", y_test, rf_pred)

Suitable Evaluation Metrics
* MAE  -> Average prediction error
* MSE  -> Penalizes large errors
* RMSE -> Easy interpretation of prediction error
* R2   -> Measures model accuracy

**5. Model Comparison**

Most reliable model:
The model with:
 1. Lowest MAE
 2. Lowest RMSE
 3. Highest R2 Score
  
  
  
  
  Usually Random Forest performs best because it handles nonlinear patterns and reduces overfitting.


**6. Insight Extraction**

Feature Importance from Random Forest

In [ ]:
importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nFEATURE IMPORTANCE")
print(importance)

Key Factors Affecting Pollution

Likely important features:
 1. traffic_density
 2. truck_count
 3. car_count
 4. wind_speed
 5. temperature

Government Preventive Actions

1. Control traffic during peak hours
 2. Restrict heavy vehicles in polluted zones
 3. Issue early pollution warnings
 4. Improve public transport
 5. Increase green zones in high pollution areas